# 05 — Matching e Inverse Probability Weighting (IPW)
**Autores clave:** Rosenbaum & Rubin (1983, 1985) · Heckman, Ichimura & Todd (1997, 1998) · Abadie & Imbens (2006, 2016) · Hirano, Imbens & Ridder (2003)

## Motivación
Bajo CIA ($\{Y(0),Y(1)\} \perp D \mid X$) con overlap, hay dos estrategias principales para estimar el ATE/ATT sin asumir un modelo paramétrico para el outcome:

1. **Matching:** emparejar cada tratado con control(es) similares en $X$ o $p(X)$
2. **IPW:** ponderar cada observación por el inverso de su propensity score

## Estimador IPW (Hirano, Imbens & Ridder, 2003)
$$\widehat{ATE}_{IPW} = \frac{1}{n}\sum_i \left[\frac{D_i Y_i}{\hat{p}(X_i)} - \frac{(1-D_i)Y_i}{1-\hat{p}(X_i)}\right]$$

## Estimador Doblemente Robusto (Robins & Rotnitzky, 1995)
$$\widehat{ATE}_{DR} = \frac{1}{n}\sum_i \left[\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i) + \frac{D_i(Y_i - \hat{\mu}_1(X_i))}{\hat{p}(X_i)} - \frac{(1-D_i)(Y_i - \hat{\mu}_0(X_i))}{1-\hat{p}(X_i)}\right]$$
Consistente si **o** el modelo de outcome **o** el propensity score está bien especificado.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import NearestNeighbors

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

n = 3000
TRUE_ATE = 2.0

# Covariables (confounders)
X1 = np.random.normal(0, 1, n)
X2 = np.random.normal(0, 1, n)

# Propensity verdadero
log_odds = -0.3 + 0.8*X1 - 0.5*X2
ps_true   = 1 / (1 + np.exp(-log_odds))
D = np.random.binomial(1, ps_true, n).astype(float)

# Outcomes potenciales
Y0 = 5 + 1.5*X1 - 0.8*X2 + np.random.normal(0, 1.5, n)
Y1 = Y0 + TRUE_ATE + 0.3*X1   # efecto heterogéneo
Y  = np.where(D==1, Y1, Y0)

df = pd.DataFrame({'Y': Y, 'D': D, 'X1': X1, 'X2': X2,
                   'Y0': Y0, 'Y1': Y1, 'ps_true': ps_true})
df['tau_i'] = Y1 - Y0

print(f'ATE verdadero: {df["tau_i"].mean():.4f}')
print(f'ATT verdadero: {df[df["D"]==1]["tau_i"].mean():.4f}')
print(f'Tratados: {D.sum():.0f} ({D.mean():.1%})')

## 1 — Nearest Neighbor Matching (Abadie & Imbens, 2006)

In [ ]:
# Estimar propensity score
lr = LogisticRegression(C=1e6, random_state=42)
lr.fit(df[['X1', 'X2']], df['D'])
df['ps_hat'] = lr.predict_proba(df[['X1', 'X2']])[:, 1]

# ── 1-to-1 Matching en propensity score (ATT) ─────────────────────────────
treated_idx = df[df['D']==1].index.values
control_idx = df[df['D']==0].index.values

ps_control = df.loc[control_idx, 'ps_hat'].values.reshape(-1, 1)
ps_treated = df.loc[treated_idx, 'ps_hat'].values.reshape(-1, 1)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(ps_control)
distances, matched_control_pos = nn.kneighbors(ps_treated)
matched_control_idx = control_idx[matched_control_pos.flatten()]

att_match = (df.loc[treated_idx, 'Y'].values -
             df.loc[matched_control_idx, 'Y'].values).mean()

# Balance antes y después
print('Balance de covariables — Antes y Después del matching')
print('─' * 65)
for col in ['X1', 'X2', 'ps_hat']:
    pre_diff  = df[df['D']==1][col].mean() - df[df['D']==0][col].mean()
    post_diff = df.loc[treated_idx, col].mean() - df.loc[matched_control_idx, col].mean()
    # Standardized Mean Difference
    pooled_sd = df[col].std()
    smd_pre   = pre_diff  / pooled_sd
    smd_post  = post_diff / pooled_sd
    print(f'  {col:<10} SMD antes: {smd_pre:>7.3f}  SMD después: {smd_post:>7.3f}  mejora: {abs(smd_pre) > abs(smd_post)}')

print(f'\nATT matching (1-to-1 en PS): {att_match:.4f}')
print(f'ATT verdadero:               {df[df["D"]==1]["tau_i"].mean():.4f}')

## 2 — Inverse Probability Weighting (IPW)

In [ ]:
# Recortar PS para estabilidad (Imbens & Rubin, 2015 recomiendan recortar en [0.01, 0.99])
eps = 0.01
ps = np.clip(df['ps_hat'], eps, 1 - eps)

# IPW Horvitz-Thompson (HT)
weights_ht = np.where(df['D']==1, 1/ps, 1/(1-ps))
ate_ht = (df['D'] * df['Y'] / ps - (1 - df['D']) * df['Y'] / (1 - ps)).mean()

# IPW Normalizado (Hájek) — más estable
w1 = df['D'] / ps; w0 = (1 - df['D']) / (1 - ps)
ate_ipw = (df['D'] * df['Y'] / ps).sum() / (df['D'] / ps).sum() - \
          ((1 - df['D']) * df['Y'] / (1 - ps)).sum() / ((1 - df['D']) / (1 - ps)).sum()

# Doblemente robusto (Robins & Rotnitzky, 1995)
lm1 = LinearRegression().fit(df[df['D']==1][['X1','X2']], df[df['D']==1]['Y'])
lm0 = LinearRegression().fit(df[df['D']==0][['X1','X2']], df[df['D']==0]['Y'])
mu1 = lm1.predict(df[['X1','X2']])
mu0 = lm0.predict(df[['X1','X2']])

dr_term = (mu1 - mu0 +
           df['D'] * (df['Y'] - mu1) / ps -
           (1 - df['D']) * (df['Y'] - mu0) / (1 - ps))
ate_dr = dr_term.mean()

# OLS simple (sesgado)
ate_ols = sm.OLS(df['Y'], sm.add_constant(df['D'])).fit().params['D']

# OLS con controles
ate_ols_x = sm.OLS(df['Y'], sm.add_constant(df[['D','X1','X2']])).fit().params['D']

results = pd.DataFrame([
    ('ATE verdadero',          df['tau_i'].mean()),
    ('OLS sin controles',      ate_ols),
    ('OLS con controles',      ate_ols_x),
    ('Matching 1-to-1 (ATT)',  att_match),
    ('IPW Horvitz-Thompson',   ate_ht),
    ('IPW Normalizado (Hájek)',ate_ipw),
    ('Doblemente Robusto',     ate_dr),
], columns=['Estimador', 'ATE estimado'])
results['sesgo'] = results['ATE estimado'] - df['tau_i'].mean()
print(results.round(4).to_string(index=False))

## 3 — Love Plot: balance de covariables

In [ ]:
# Calcular SMD para varias estrategias
covs = ['X1', 'X2']

def smd(treated_vals, control_vals):
    pooled_sd = np.sqrt((treated_vals.var() + control_vals.var()) / 2)
    return (treated_vals.mean() - control_vals.mean()) / pooled_sd

# IPW pesos para el balance
w_ipw = np.where(df['D']==1, 1/ps, 1/(1-ps))

smd_rows = []
for col in covs:
    t_vals = df[df['D']==1][col]; c_vals = df[df['D']==0][col]
    smd_rows.append({
        'variable': col,
        'Unadjusted':   smd(t_vals, c_vals),
        'Matching 1:1': smd(df.loc[treated_idx, col],
                            df.loc[matched_control_idx, col]),
        'IPW':          smd(df[df['D']==1][col],
                            pd.Series(w_ipw[df['D']==0] * df[df['D']==0][col]) /
                            w_ipw[df['D']==0].sum() * len(df[df['D']==0]) - t_vals.mean()),
    })

smd_df = pd.DataFrame(smd_rows)

fig, ax = plt.subplots(figsize=(9, 4))
methods = ['Unadjusted', 'Matching 1:1']
colors  = ['#f72585', '#4361ee']
markers = ['X', 'o']
y_pos   = np.arange(len(covs))

for method, color, marker in zip(methods, colors, markers):
    ax.scatter(smd_df[method], y_pos, color=color, s=120, marker=marker,
               zorder=5, label=method)

ax.axvline(0,    color='black', linewidth=1)
ax.axvline(0.1,  color='#aaa', linewidth=1, linestyle='--', label='|SMD| = 0.1')
ax.axvline(-0.1, color='#aaa', linewidth=1, linestyle='--')
ax.set_yticks(y_pos); ax.set_yticklabels(covs)
ax.set_xlabel('Standardized Mean Difference (SMD)')
ax.set_title('Love Plot — Balance de Covariables\n|SMD| < 0.1 = balance aceptable (Imbens & Rubin, 2015)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Resumen

| Estimador | Supuestos | Ventajas | Limitaciones |
|---|---|---|---|
| OLS con controles | CIA + linealidad | Simple | Sesgo por mala especificación |
| Matching 1:1 | CIA + overlap | No paramétrico | Alta varianza con pocos controles |
| IPW | CIA + overlap | Ponderación flexible | Inestable con PS extremos |
| Doblemente Robusto | CIA + overlap | Un modelo puede fallar | Más complejo |

**Referencias:**
- Rosenbaum, P.R. & Rubin, D.B. (1983). The central role of the propensity score. *Biometrika*, 70(1).
- Heckman, J.J., Ichimura, H. & Todd, P. (1997). Matching as an econometric evaluation estimator. *Review of Economic Studies*, 65(2).
- Hirano, K., Imbens, G.W. & Ridder, G. (2003). Efficient estimation of average treatment effects. *Econometrica*, 71(4).
- Abadie, A. & Imbens, G.W. (2006). Large sample properties of matching estimators. *Econometrica*, 74(1).
- Robins, J.M. & Rotnitzky, A. (1995). Semiparametric efficiency in multivariate regression models. *JASA*, 90(429).

**Siguiente:** `06_synthetic_control.ipynb`